# Лабораторна робота №3

## Візуалізація даних

**Студент:** Сапронов Анатолій  
**Група:** ФБ-45  
**Дисципліна:** Засоби підготовки та аналізу даних

## Мета роботи

Метою роботи є вибір датасету, очищення даних та побудова візуалізацій, які показують залежності між ознаками й корисну інформацію про набір даних.


## 1. Вибір датасету

Для лабораторної роботи обрано **Adult Dataset** з UCI Machine Learning Repository.

Датасет підходить під вимоги лабораторної роботи, тому що:
- він є multivariate;
- містить категоріальні, integer та real attributes;
- має більше двох числових атрибутів;
- містить missing values, які позначені символом `?`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from pandas.plotting import parallel_coordinates, scatter_matrix

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
DATA_PATH = DATA_DIR / "adult.data"

columns = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education_num",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital_gain",
    "capital_loss",
    "hours_per_week",
    "native_country",
    "income"
]


## 2. Завантаження даних

Дані завантажуються автоматично з UCI. Якщо файл уже є в папці `data`, повторне завантаження не потрібне.


In [ ]:
def download_dataset(url: str, path: Path) -> Path:
    if path.exists():
        print("Файл уже існує, повторне завантаження не потрібне.")
        return path

    df_online = pd.read_csv(url, header=None, names=columns, na_values="?", skipinitialspace=True)
    df_online.to_csv(path, index=False)
    print(f"Датасет збережено у файл: {path}")
    return path


download_dataset(DATA_URL, DATA_PATH)


In [ ]:
df = pd.read_csv(DATA_PATH)
df.head()


## 3. Первинний огляд датасету

Перевіримо розмір датасету, типи даних та кількість пропущених значень.


In [ ]:
print("Розмір датасету:", df.shape)
df.info()


In [ ]:
df.isna().sum()


## 4. Data Cleaning

У цьому датасеті пропущені значення позначаються як `?`.  
Після завантаження вони вже перетворені в `NaN`.

Під час очищення:
- видаляємо дублікати;
- прибираємо рядки з пропущеними значеннями;
- очищуємо текстові значення від зайвих пробілів;
- перевіряємо коректність числових колонок.


In [ ]:
def clean_adult_dataset(data: pd.DataFrame) -> pd.DataFrame:
    cleaned = data.copy()

    # Прибираємо зайві пробіли у текстових колонках
    object_cols = cleaned.select_dtypes(include="object").columns
    for col in object_cols:
        cleaned[col] = cleaned[col].astype(str).str.strip()
        cleaned[col] = cleaned[col].replace("?", np.nan)

    # Видаляємо дублікати та пропущені значення
    cleaned = cleaned.drop_duplicates()
    cleaned = cleaned.dropna().reset_index(drop=True)

    # Перетворення числових колонок
    numeric_cols = ["age", "fnlwgt", "education_num", "capital_gain", "capital_loss", "hours_per_week"]
    for col in numeric_cols:
        cleaned[col] = pd.to_numeric(cleaned[col], errors="coerce")

    cleaned = cleaned.dropna().reset_index(drop=True)

    return cleaned


clean_df = clean_adult_dataset(df)

print("Розмір до очищення:", df.shape)
print("Розмір після очищення:", clean_df.shape)
clean_df.head()


In [ ]:
clean_df.isna().sum()


## 5. Описова статистика

Перед побудовою графіків подивимося на основні статистичні показники числових атрибутів.


In [ ]:
clean_df.describe()


## 6. Scatter plot: залежність віку від кількості годин роботи на тиждень

Цей графік показує, як пов'язані між собою два числові атрибути: `age` та `hours_per_week`.


In [ ]:
plt.figure(figsize=(9, 6))
plt.scatter(clean_df["age"], clean_df["hours_per_week"], alpha=0.25)
plt.title("Залежність hours_per_week від age")
plt.xlabel("Age")
plt.ylabel("Hours per week")
plt.grid(True, alpha=0.3)
plt.show()


## 7. Line plot: середня кількість робочих годин за віком

Для більш зрозумілого представлення даних згрупуємо записи за віком і покажемо середнє значення `hours_per_week`.


In [ ]:
age_hours = clean_df.groupby("age")["hours_per_week"].mean().reset_index()

plt.figure(figsize=(10, 6))
plt.plot(age_hours["age"], age_hours["hours_per_week"], marker="o", markersize=3)
plt.title("Середня кількість робочих годин за віком")
plt.xlabel("Age")
plt.ylabel("Average hours per week")
plt.grid(True, alpha=0.3)
plt.show()


## 8. Histogram: розподіл віку

Гістограма показує кількість людей у п'яти заданих діапазонах віку.


In [ ]:
plt.figure(figsize=(9, 6))
plt.hist(clean_df["age"], bins=5, edgecolor="black")
plt.title("Розподіл віку у 5 діапазонах")
plt.xlabel("Age")
plt.ylabel("Number of records")
plt.grid(True, alpha=0.3)
plt.show()


## 9. Bar chart: кількість записів за рівнем доходу

Цей графік показує, скільки записів належить до кожної категорії доходу.


In [ ]:
income_counts = clean_df["income"].value_counts()

plt.figure(figsize=(7, 5))
plt.bar(income_counts.index, income_counts.values)
plt.title("Кількість записів за рівнем доходу")
plt.xlabel("Income")
plt.ylabel("Count")
plt.grid(axis="y", alpha=0.3)
plt.show()


## 10. Box plot: робочі години за категоріями доходу

Box plot дозволяє порівняти розподіл `hours_per_week` для двох груп доходу.


In [ ]:
groups = [
    clean_df.loc[clean_df["income"] == "<=50K", "hours_per_week"],
    clean_df.loc[clean_df["income"] == ">50K", "hours_per_week"]
]

plt.figure(figsize=(8, 6))
plt.boxplot(groups, labels=["<=50K", ">50K"])
plt.title("Розподіл hours_per_week за рівнем доходу")
plt.xlabel("Income")
plt.ylabel("Hours per week")
plt.grid(True, alpha=0.3)
plt.show()


## 11. Correlation heatmap

Теплова карта кореляції показує зв'язки між числовими атрибутами датасету.


In [ ]:
numeric_cols = ["age", "fnlwgt", "education_num", "capital_gain", "capital_loss", "hours_per_week"]
corr = clean_df[numeric_cols].corr()

plt.figure(figsize=(9, 7))
plt.imshow(corr)
plt.colorbar(label="Correlation")
plt.xticks(range(len(numeric_cols)), numeric_cols, rotation=45, ha="right")
plt.yticks(range(len(numeric_cols)), numeric_cols)
plt.title("Correlation heatmap")

for i in range(len(numeric_cols)):
    for j in range(len(numeric_cols)):
        plt.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center")

plt.tight_layout()
plt.show()


## 12. Багатовимірна візуалізація: Parallel Coordinates Plot

Це один із прикладів багатовимірної візуалізації.  
Щоб графік був читабельним, використовується випадкова вибірка з очищеного датасету.

Для графіка взято числові атрибути:
- `age`
- `education_num`
- `capital_gain`
- `capital_loss`
- `hours_per_week`

Категорією для порівняння є `income`.


In [ ]:
parallel_df = clean_df[["age", "education_num", "capital_gain", "capital_loss", "hours_per_week", "income"]].sample(
    n=min(300, len(clean_df)),
    random_state=42
).copy()

# Нормалізація числових колонок, щоб вони були в одному масштабі
parallel_numeric = ["age", "education_num", "capital_gain", "capital_loss", "hours_per_week"]
for col in parallel_numeric:
    min_value = parallel_df[col].min()
    max_value = parallel_df[col].max()
    parallel_df[col] = (parallel_df[col] - min_value) / (max_value - min_value)

plt.figure(figsize=(12, 6))
parallel_coordinates(parallel_df, class_column="income", alpha=0.35)
plt.title("Parallel Coordinates Plot для числових ознак")
plt.xlabel("Attributes")
plt.ylabel("Normalized value")
plt.grid(True, alpha=0.3)
plt.show()


## 13. Scatter matrix

Scatter matrix також допомагає побачити взаємозв'язки між кількома числовими атрибутами одразу.


In [ ]:
matrix_df = clean_df[["age", "education_num", "hours_per_week", "capital_gain"]].sample(
    n=min(500, len(clean_df)),
    random_state=42
)

scatter_matrix(matrix_df, figsize=(10, 10), diagonal="hist", alpha=0.4)
plt.suptitle("Scatter matrix для числових атрибутів", y=1.02)
plt.show()


## 14. Додаткова корисна інформація: середній вік за освітою

Цей графік показує топ-10 рівнів освіти за середнім віком людей у датасеті.


In [ ]:
education_age = (
    clean_df.groupby("education")["age"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
)

plt.figure(figsize=(10, 6))
plt.barh(education_age.index, education_age.values)
plt.title("Топ-10 рівнів освіти за середнім віком")
plt.xlabel("Average age")
plt.ylabel("Education")
plt.grid(axis="x", alpha=0.3)
plt.show()


## Висновок

У лабораторній роботі було обрано Adult Dataset, який відповідає заданим вимогам.  
Після цього було виконано data cleaning: видалено дублікати, пропущені значення та очищено текстові атрибути.

У роботі побудовано 8 графіків, серед яких є:
- scatter plot;
- line plot;
- histogram;
- bar chart;
- box plot;
- correlation heatmap;
- parallel coordinates plot;
- scatter matrix.

Побудовані графіки дозволяють краще зрозуміти структуру датасету, розподіл числових атрибутів і залежності між ними.
